# Customer Churn Dataset — Data Cleaning

**Objective:** Convert the raw customer churn CSV into a consistent, analysis-ready dataset using Pandas.

## Cleaning decisions
1. Remove exact duplicate rows. In this dataset, repeated Customer IDs were exact copies.
2. Convert `total_charges_inr` to numeric. The text value `unknown` is treated as missing.
3. Fill missing `region` values with the explicit category `Unknown`; this avoids assigning customers to an unsupported region.
4. For missing total charges where tenure is zero, use `0` because no subscription months have elapsed. For the remaining missing totals, use the median of observed total charges to retain records without letting a few extreme values dominate the imputation.
5. Standardize column names to lowercase snake_case and trim/canonicalize categorical text.
6. Validate numeric ranges. Unusual values are flagged rather than silently changed when there is no defensible correction rule.
7. No date column exists in the source file, so no date conversion is applicable.

Run this notebook in the same folder as `customer_churn_raw_v2.csv`.

In [ ]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

raw_file = Path("customer_churn_raw_v2.csv")
df = pd.read_csv(raw_file)

print("Rows and columns:", df.shape)
display(df.head())
print("\\nData types:")
display(df.dtypes)
print("\\nMissing values:")
display(df.isna().sum())
print("\\nExact duplicate rows:", df.duplicated().sum())

## 1. Remove exact duplicates and standardize headers

In [ ]:
rows_before = len(df)
duplicates_before = int(df.duplicated().sum())
df = df.drop_duplicates().copy()

def snake_case(name):
    name = re.sub(r"([a-z0-9])([A-Z])", r"\\1_\\2", str(name))
    name = re.sub(r"[^A-Za-z0-9]+", "_", name)
    return name.strip("_").lower()

df.columns = [snake_case(c) for c in df.columns]
print(f"Removed {duplicates_before} exact duplicate rows.")
print("Rows remaining:", len(df))
print("Standardized columns:", df.columns.tolist())

## 2. Normalize text and convert data types

In [ ]:
text_cols = df.select_dtypes(include="object").columns.tolist()
for col in text_cols:
    df[col] = df[col].map(
        lambda x: re.sub(r"\\s+", " ", x.strip()) if isinstance(x, str) else x
    )

numeric_cols = [
    "age", "tenure_months", "monthly_charges_inr", "total_charges_inr",
    "support_tickets", "late_payments"
]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

categorical_cols = [
    "region", "subscription_type", "contract_type",
    "payment_method", "auto_pay", "churn"
]
for col in categorical_cols:
    df[col] = df[col].map(
        lambda x: x.title() if isinstance(x, str) and x.strip() else x
    )

print("Data types after conversion:")
display(df.dtypes)
print("Missing values after conversion:")
display(df.isna().sum())

## 3. Handle missing values

In [ ]:
missing_region = int(df["region"].isna().sum())
df["region"] = df["region"].fillna("Unknown")

missing_total_before = int(df["total_charges_inr"].isna().sum())
zero_tenure_unknown = df["total_charges_inr"].isna() & (df["tenure_months"] == 0)
df.loc[zero_tenure_unknown, "total_charges_inr"] = 0.0

remaining_missing_total = int(df["total_charges_inr"].isna().sum())
total_charge_median = float(df["total_charges_inr"].median())
df["total_charges_inr"] = df["total_charges_inr"].fillna(total_charge_median)

print("Missing regions replaced with 'Unknown':", missing_region)
print("Missing total-charge values before imputation:", missing_total_before)
print("Zero-tenure missing totals set to 0:", int(zero_tenure_unknown.sum()))
print("Other missing totals filled with median:", remaining_missing_total)
print("Median used (INR):", round(total_charge_median, 2))
print("Remaining null values:", int(df.isna().sum().sum()))

## 4. Validate values and export the cleaned CSV

In [ ]:
validation = {
    "age_out_of_range_(expected_18_to_100)": int((~df["age"].between(18, 100)).sum()),
    "negative_tenure": int((df["tenure_months"] < 0).sum()),
    "negative_monthly_charges": int((df["monthly_charges_inr"] < 0).sum()),
    "negative_total_charges": int((df["total_charges_inr"] < 0).sum()),
    "negative_support_tickets": int((df["support_tickets"] < 0).sum()),
    "negative_late_payments": int((df["late_payments"] < 0).sum()),
}
display(pd.Series(validation, name="Invalid record count"))

df["monthly_charges_inr"] = df["monthly_charges_inr"].round(2)
df["total_charges_inr"] = df["total_charges_inr"].round(2)

print("Remaining null values:", int(df.isna().sum().sum()))
print("Remaining exact duplicate rows:", int(df.duplicated().sum()))
print("Final shape:", df.shape)

clean_file = Path("customer_churn_cleaned.csv")
df.to_csv(clean_file, index=False)
print("Saved:", clean_file.resolve())
display(df.head())

## Final result

The exported `customer_churn_cleaned.csv` is the analysis-ready dataset. The notebook documents the cleaning logic and can be rerun against the original CSV. Review any validation counts before modeling; this workflow intentionally does not delete unusual numeric observations without evidence that they are errors.